# 1 – InterGATE: Results & Bootstrap Analysis

Post-hoc analysis pipeline: carga un bundle entrenado y ejecuta:
- Evaluación con umbrales OVR y bias por clase
- Bootstrap 95% CI (hard labels + AUC OVR)
- Explicabilidad Grad×Input
- Enrichment (gProfiler / Enrichr / KEGG overlay)
- Análisis de ejes biológicos


## 0) Imports

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ── Core pipeline ──
from intergate.config import CFG
from intergate.utils import set_seed, set_all_seeds, cleanup_memory, get_device
from intergate.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only,
    make_dataloaders, build_regulator_features,
)
from intergate.graph import build_backbone
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.losses import compute_metrics_full, compute_class_weights_balanced
from intergate.training import predict_proba_xh_mode
from intergate.pruning import export_pruned_graph, evaluate_keep_ratios
from intergate.stability import edge_set_from_edge_index, pairwise_jaccard_stats

# ── Ablation / bundle ──
from intergate.ablation import (
    AblationConfig, load_bundle, build_pruned_model_from_bundle,
    register_runtime, get_full_gene_matrix_and_genes,
)

# ── Post-processing ──
from intergate.postprocessing import (
    ovr_thresholds_from_val, predict_with_ovr_thresholds,
    plot_confusion_matrix,
    gene_importance_grad_x_input, aggregate_gene_importance,
)

# ── Bootstrap ──
from intergate.bootstrap import (
    scores_from_proba, tune_class_biases,
    bootstrap_classification_ci, bootstrap_auc_ovr_ci,
)

# ── Visualization ──
from intergate.visualization import (
    plot_pruned_subgraph_all_components, plot_components_grid,
    draw_global_graph_from_bundle,
)

# ── Axes ──
from intergate.axes import (
    compute_axis_scores, plot_axis_boxplots,
    plot_axis_heatmap_by_subtype, plot_axis_correlation,
    plot_axis_embedding, axis_stats_by_subtype,
    one_vs_rest_axis_analysis, summarize_one_vs_rest,
    plot_one_vs_rest_heatmap,
)

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


## 1) Configuración

Overrides opcionales. Ajusta rutas si es necesario.

In [ ]:
# CFG.DATA_DIR ya apunta por defecto a PROJECT_ROOT/data/processed.
# CFG.ARTIFACTS_ROOT ya apunta por defecto a PROJECT_ROOT/artifacts_ablation.
print("Config OK")
print("DATA_DIR:", CFG.DATA_DIR)
print("EXPR_CSV:", CFG.EXPR_CSV)
print("META_CSV:", CFG.META_CSV)


## 2) Data pipeline (datos + grafo + split + escalado)

In [ ]:
# ── Cargar expresión y metadata ──
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)


In [ ]:
# ── Backbone graph (con caché) ──
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False,
    use_omnipath=CFG.USE_OMNIPATH,
    use_huri=CFG.USE_HURI,
)
print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


In [ ]:
# ── Split + escalado + connected-only ──
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )


In [ ]:
# ── Regulator features (con caché) ──
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    stats=CFG.REG_STATS,
    min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)
print(f"Xs_graph: {Xs_graph.shape}")


In [ ]:
# ── DataLoaders ──
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx,
    batch_size=CFG.BATCH_SIZE,
)


In [ ]:
# ── Tensores de grafo + registro en runtime ──
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg,
    X_h=Xs_graph, y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t,
    edge_type_t=edge_type_t,
)


## 3) Cargar bundle y modelo pre-entrenado

In [ ]:
BUNDLE_DIR = os.path.join(CFG.ARTIFACTS_ROOT, "FULL", "seed_1234")
bundle = load_bundle(BUNDLE_DIR)

meta_b = bundle.get("meta", {})
splits = meta_b.get("splits", {})
g = bundle["graph"]
print("Bundle cargado:", list(bundle.keys()))


In [ ]:
# ── Reconstruir espacio compacto del modelo ──
nodes_used = np.asarray(g["nodes_used_full"], dtype=np.int64)
X_full, genes_full = get_full_gene_matrix_and_genes()
genes_model = [str(genes_full[i]) for i in nodes_used]
Xs_gene_compact = X_full[:, nodes_used]

print(f"Modelo compacto: {len(genes_model)} genes, Xs_gene_compact: {Xs_gene_compact.shape}")


In [ ]:
# ── Cargar modelo con pesos ──
model_loaded = build_pruned_model_from_bundle(bundle, device=DEVICE)
model_loaded.eval()
print("Modelo cargado:", type(model_loaded).__name__)


In [ ]:
# ── Dataloaders compactos ──
dl_va_c, dl_te_c = [
    torch.utils.data.DataLoader(
        __import__("intergate.data", fromlist=["ExpressionDataset"]).ExpressionDataset(
            Xs_gene_compact, Xs_graph, y, idx
        ),
        batch_size=CFG.BATCH_SIZE, shuffle=False,
    )
    for idx in [val_idx, test_idx]
]

# ── Predicciones ──
proba_va, y_va = predict_proba_xh_mode(model_loaded, None, dl_va_c, DEVICE, xh_mode="orig")
proba_te, y_te = predict_proba_xh_mode(model_loaded, None, dl_te_c, DEVICE, xh_mode="orig")
print(f"proba_va: {proba_va.shape}, proba_te: {proba_te.shape}")


## 4) Grafo podado: nodos, grados, export

In [ ]:
# ── Extraer edge_index del bundle ──
edge_index_p = g.get("edge_index_compact", g.get("edge_index_comp"))
edge_weight_p = g.get("edge_weight_compact", g.get("edge_weight_comp"))
if torch.is_tensor(edge_index_p):
    edge_index_p = edge_index_p.cpu()
    edge_weight_p = edge_weight_p.cpu() if torch.is_tensor(edge_weight_p) else edge_weight_p

n_total = len(genes_model)
node_ids = torch.unique(edge_index_p.reshape(-1)).cpu().numpy()
node_ids = np.sort(node_ids)
genes_used = [genes_model[i] for i in node_ids]

print(f"[NODES total model]: {n_total}")
print(f"[NODES used in pruned graph]: {len(node_ids)}")

# Grados
deg = np.zeros(n_total, dtype=int)
src, dst = edge_index_p[0].cpu().numpy(), edge_index_p[1].cpu().numpy()
for u in src: deg[u] += 1
for v in dst: deg[v] += 1

df_deg = pd.DataFrame({"node_id": np.arange(n_total), "gene": genes_model, "degree_in_pruned": deg})
df_deg_used = df_deg[df_deg["degree_in_pruned"] > 0].sort_values("degree_in_pruned", ascending=False)
print("Top genes by degree:")
display(df_deg_used.head(20))


In [ ]:
# ── Gene symbols del subgrafo + máscara ──
gene_symbols_pruned = (
    df_deg_used["gene"].astype(str).str.strip()
    .replace("", np.nan).dropna().drop_duplicates().tolist()
)
G_total = Xs_gene_compact.shape[1]
pruned_ids = df_deg_used["node_id"].astype(int).to_numpy()
gene_mask_pruned = np.zeros(G_total, dtype=bool)
gene_mask_pruned[pruned_ids] = True

print(f"Gene symbols únicos: {len(gene_symbols_pruned)}")
print(f"Genes en máscara pruned: {gene_mask_pruned.sum()}")

# Para enrichment / KEGG
genes_ckpt = list(gene_symbols_pruned)


## 5) Visualización del grafo

In [ ]:
# ── Componentes del subgrafo ──
G_all, pos_all, comps_all = plot_pruned_subgraph_all_components(edge_index_p, edge_weight_p)
plot_components_grid(G_all, genes=genes_model)


In [ ]:
# ── Grafo dirigido completo desde bundle ──
draw_global_graph_from_bundle(
    bundle,
    title="Gene Regulatory Network",
    layout="kamada_kawai", seed=42,
    figsize=(18, 15), facecolor="#0d1117",
    node_color_mode="degree", node_cmap=plt.cm.plasma,
    node_size_mode="degree", node_size_base=60.0, node_size_scale=1200.0,
    node_alpha=0.92, node_edgecolor="#ffffff", node_linewidth=0.6,
    top_labels=40, label_fontsize=8, label_fontweight="bold", label_bbox_alpha=0.0,
    edge_color_mode="weight", edge_cmap=plt.cm.cool,
    edge_width_mode="weight", edge_width_min=0.3, edge_width_max=2.8, edge_alpha=0.35,
    arrows=True, arrowstyle="-|>", arrowsize=16,
    connectionstyle="arc3,rad=0.08",
    show_colorbar_nodes=False, show_colorbar_edges=False,
    annotate_stats=False,
    save_path="./figures_axes_refined/Grafo_final.png", dpi=300,
)


## 6) Classification report + umbrales OVR

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

num_classes = proba_te.shape[1]

# ── OVR thresholds optimizados en validación ──
thr_ovr = ovr_thresholds_from_val(proba_va, y_va, num_classes, grid=np.linspace(0.01, 0.99, 300))
y_pred_ovr = predict_with_ovr_thresholds(proba_te, thr_ovr)

print(classification_report(y_te, y_pred_ovr, target_names=classes, digits=3))
_ = plot_confusion_matrix(y_te, y_pred_ovr, classes, title="Matriz de Confusión - OVR")


## 7) Calibración por class bias (greedy)

In [ ]:
scores_val = scores_from_proba(proba_va)
scores_te = scores_from_proba(proba_te)

best_bias, best_f1_bias = tune_class_biases(
    scores_val, y_va, classes=classes,
    n_passes=3, verbose=True,
)

from intergate.bootstrap import predict_with_class_bias
y_pred_bias = predict_with_class_bias(scores_te, best_bias)

print("\n=== Test con class biases ===")
print(classification_report(y_te, y_pred_bias, target_names=classes, digits=3))
_ = plot_confusion_matrix(y_te, y_pred_bias, classes, title="Matriz de Confusión - Class Bias")


## 8) Bootstrap 95% CI

### 8a) Bootstrap métricas hard-label

In [ ]:
# Usa la predicción final que prefieras (OVR o class-bias)
y_pred_final = y_pred_ovr

boot_ci = bootstrap_classification_ci(
    y_true=y_te, y_pred=y_pred_final,
    class_names=classes, B=10_000, seed=42,
)

print("=== Bootstrap CI (overall) ===")
display(boot_ci["overall"])
print("=== Bootstrap CI (per-class) ===")
display(boot_ci["per_class"])
print("=== Observed confusion matrix ===")
display(boot_ci["observed_confusion_matrix"])

# Guardar
os.makedirs("./Results", exist_ok=True)
boot_ci["overall"].to_csv("./Results/bootstrap_ci_overall_test.csv", index=False)
boot_ci["per_class"].to_csv("./Results/bootstrap_ci_per_class_test.csv", index=False)


### 8b) Bootstrap AUC OVR

In [ ]:
boot_auc = bootstrap_auc_ovr_ci(
    y_true=y_te, proba=proba_te,
    class_names=classes, B=10_000, seed=42,
)

print("=== AUC Bootstrap CI (overall) ===")
display(boot_auc["macro"])
print("=== AUC Bootstrap CI (per-class) ===")
display(boot_auc["per_class"])

boot_auc["macro"].to_csv("./Results/bootstrap_auc_overall.csv", index=False)
boot_auc["per_class"].to_csv("./Results/bootstrap_auc_per_class.csv", index=False)


## 9) Explicabilidad: Grad×Input

In [ ]:
assert Xs_gene_compact.shape[1] == len(genes_model), "Xs_gene_compact no coincide con genes_model"

Xg_test = Xs_gene_compact[test_idx]
Xh_test = Xs_graph[test_idx]
y_test = y[test_idx]

imp_by_class = {}
for c in range(n_classes):
    imp_by_class[c] = aggregate_gene_importance(
        model_loaded, Xg_test, Xh_test, y_test,
        class_id=c, device=DEVICE, max_samples=200,
        gene_mask=gene_mask_pruned, zero_outside=True,
    )


In [ ]:
TOPK = 500
use_idx = np.where(gene_mask_pruned)[0]

top_genes_by_class = {}
for c in range(n_classes):
    scores = imp_by_class[c][use_idx]
    order_local = np.argsort(-scores)[:min(TOPK, scores.size)]
    top_global = use_idx[order_local]
    top_genes_by_class[c] = [(genes_model[i], float(imp_by_class[c][i])) for i in top_global]

# Top-10 por clase
for c in range(n_classes):
    print(f"\n--- {classes[c]} ---")
    print(top_genes_by_class[c][:10])


In [ ]:
# Guardar importancias de la clase TARGET
TARGET_CLASS_ID = label_map.get("TNBC", 0)
pd.DataFrame(top_genes_by_class[TARGET_CLASS_ID], columns=["gene", "importance"]).to_csv(
    "./Results/genes_grafo_TARGET_Importancia.csv", index=False
)
df_deg_used.to_csv("./Results/genes_grafo.csv")


## 10) Enrichment (gProfiler / Enrichr / KEGG)

Requiere internet y paquetes opcionales: `gprofiler-official`, `gseapy`, `mygene`.

In [ ]:
import re

gene_list = [str(g).strip().upper() for g in gene_symbols_pruned]
keep_tag = "pruned"
OUT_PREFIX = f"enrich_keep_{keep_tag}"
print(f"Genes para enrichment: {len(gene_list)}")

# ── gProfiler (GO + Reactome + KEGG) ──
try:
    from gprofiler import GProfiler
    gp = GProfiler(return_dataframe=True)
    res = gp.profile(organism="hsapiens", query=gene_list,
                     sources=["GO:BP", "GO:MF", "GO:CC", "REAC", "KEGG"])
    display(res.head(20))
    res.to_csv(f"./Results/{OUT_PREFIX}_gprofiler_GO_REAC_KEGG.csv", index=False)
except Exception as e:
    print("[WARN] gprofiler:", e)

# ── Enrichr via gseapy (fallback) ──
try:
    import gseapy as gpy
    libs = gpy.get_library_name()
    kegg_libs = [l for l in libs if re.match(r"^KEGG_\d{4}_Human$", l)]
    kegg_lib = sorted(kegg_libs, key=lambda x: int(x.split("_")[1]))[-1] if kegg_libs else "KEGG_2021_Human"

    for gs, tag in [(kegg_lib, "KEGG"), ("Reactome_2022", "REACTOME"),
                    ("GO_Biological_Process_2023", "GO_BP"),
                    ("GO_Molecular_Function_2023", "GO_MF"),
                    ("GO_Cellular_Component_2023", "GO_CC")]:
        enr = gpy.enrichr(gene_list=gene_list, gene_sets=[gs], organism="human", outdir=None, cutoff=0.5)
        df = enr.results.copy() if hasattr(enr, "results") else pd.DataFrame()
        if not df.empty:
            df.to_csv(f"./Results/{OUT_PREFIX}_{tag}.csv", index=False)
            display(df.head(10))
except Exception as e:
    print("[WARN] gseapy:", e)


### KEGG pathway overlay

In [ ]:
try:
    import mygene
    from intergate.enrichment import build_symbol_entrez_maps, plot_kegg_map_overlay

    symbol_to_entrez, entrez_to_symbol = build_symbol_entrez_maps(genes_ckpt)
    unmapped = [g for g in gene_list if g not in symbol_to_entrez]
    print(f"Unmapped symbols: {len(unmapped)}")
except ImportError:
    print("[WARN] mygene no disponible. Instala: pip install mygene")


In [ ]:
# Descomenta la ruta KEGG que quieras visualizar
try:
    for pathway in ["hsa05224", "hsa04110", "hsa05200"]:
        plot_kegg_map_overlay(
            pathway,
            selected_genes=gene_symbols_pruned,
            symbol_to_entrez=symbol_to_entrez,
            entrez_to_symbol=entrez_to_symbol,
            show_labels=True,
        )
except Exception as e:
    print("[WARN] KEGG overlay:", e)


## 11) Análisis de ejes biológicos

In [ ]:
AXES = {
    "Luminal Hormonal": [
        "ESR1","PGR","GATA3","GREB1","ESR2","PHLDA1","LRIG1","RXRA"
    ],
    "Cell Cycle Mitotic": [
        "AURKA","AURKB","BUB1","CDC7","CDC23","CDCA3","CDK1","CDK2",
        "CEP55","E2F1","FOXM1","PLK1","TTK","ZWINT"
    ],
    "HER2 RTK MAPK": [
        "ERBB2","EGFR","IGF1R","MET","PDGFRB","AKT1","MAP2K1","MAP2K2",
        "MAPK1","MAPK3","MAPK14","PLCG1","SRC","LYN"
    ],
    "Basal Plasticity TNBC": [
        "KRT6A","KRT16","SOX10","VGLL1","VGLL3","EGFR","EMP1","MSLN"
    ],
    "Immune Lymphoid Signaling": [
        "CD79A","CXCL9","DEF6","GRAP2","HCK","JAK3","ITPKB","PAX5",
        "SYK","S1PR1","SOCS1","TRAF1","TRAF2","ZBP1"
    ],
    "DNA Damage p53 Checkpoint": [
        "ATM","ERCC3","FANCG","KAT5","PRKDC","TP53","MDM2","PTEN",
        "DAPK1","STK11","SIRT1"
    ],
    "Adhesion Cytoskeleton Invasion": [
        "ABI2","FLNA","ITGB1","ITGB1BP1","MMP2","PTK2","RHOA","ROCK1",
        "LASP1","NCK1","NCK2","SDCBP"
    ],
    "Androgen Apocrine": [
        "AR","MSLN"
    ],
}
AXES_ORDER = list(AXES.keys())

In [ ]:
# ── Cargar expresión cruda (genes × samples) para ejes ──
df_expr_raw = pd.read_csv(CFG.EXPR_CSV, index_col=0)
metadata_axes = pd.read_csv(CFG.META_CSV, index_col=0)
metadata_axes = metadata_axes[["sample", "batch", "label"]]

meta_scores, axis_info, Xz = compute_axis_scores(
    expr_df=df_expr_raw, meta_df=metadata_axes,
    axes=AXES, sample_col="sample", label_col="label",
)
display(axis_info[["Axis", "n_present", "coverage", "present_genes"]])


In [ ]:
plot_axis_boxplots(meta_scores, label_col="label")
heat = plot_axis_heatmap_by_subtype(meta_scores, label_col="label")
display(heat)
corr = plot_axis_correlation(meta_scores)
display(corr)
plot_axis_embedding(meta_scores, label_col="label")
stats_df = axis_stats_by_subtype(meta_scores, label_col="label")
display(stats_df)


In [ ]:
# ── One-vs-rest por subtipo ──
axes_order = list(AXES.keys())
ovr_df = one_vs_rest_axis_analysis(meta_scores, label_col="label", axes_order=axes_order)
display(ovr_df.head(20))

ovr_summary = summarize_one_vs_rest(ovr_df, top_n=3, sort_by="delta_mean")
display(ovr_summary)

plot_one_vs_rest_heatmap(ovr_df, value_col="delta_mean", fdr_col="p_fdr_subtype")


# Fin